In [ ]:
# A1 · Part 2 — Build a minimal retrieval system.  Starter notebook.
# PE6201 · Emerging AI Technologies. Run top to bottom, then replace the placeholder domain.

# A1 · Part 2 — Build a minimal retrieval system  *(starter)*

**What Part 2 asks of you.** Assemble a small collection of documents from **your** domain —
10 to 30 is enough. Build a system that retrieves relevant passages and uses them to answer
questions. Show it answering **five questions it could not possibly answer without your
documents**. Then **break it**: find a question where retrieval fails or returns the wrong
passage, and explain *why* it failed.

> **The breaking is not optional and it is not a penalty.** Finding the edge of your own system
> is the single most valuable thing you will do in this assignment. A notebook where everything
> works is a notebook that was not pushed hard enough.

**What this notebook is.** A working retrieval-augmented generation (RAG) pipeline on a
**placeholder domain** (houseplant care notes). Everything you must change is marked
**`### REPLACE ME`**.

**Cost.** About 10 short generation calls. Fractions of a cent. Embeddings run locally and free.

**Deliverable from this notebook:** this notebook (run, output visible), five worked examples,
one documented failure, and roughly 400 words.

> **If a cell does not work.** This notebook was written before you ran it, against library
> versions, model names and provider behaviour that change without notice — so treat it as a
> working draft, not gospel. **Read the error, then try to fix it yourself:** debugging is part of
> the work, not a detour from it. Then **report it** — in class or by email — with the cell, the
> full error text, and what you tried, so the shared copy gets fixed and nobody behind you loses
> the same hour. Reporting a problem costs you nothing. The one thing not to do is quietly work
> around a broken cell: if your fix changes what the notebook measures, say so in your writeup.

## Final-project revision after instructor feedback

The final system uses this retrieval layer to ground training advice. The original wrist-soreness failure is retained because it cleanly demonstrates a retrieval-ranking failure with safe generation abstention.


### Setup

In [4]:
%pip -q install openai scikit-learn sentence-transformers

### Connect to OpenRouter
The **generation** step (writing the answer) goes to OpenRouter with your key. The
**embedding** step (turning text into vectors) runs **locally and free** — OpenRouter serves chat
models, not embedding models, which is exactly the "which layer do you rent?" question from
Class 2.

In [5]:
import os, getpass, time
import numpy as np
from openai import OpenAI

def _get_openrouter_key():
    k = os.environ.get('MY_PRIVATE_OPENROUTER_KEY') or os.environ.get('OPENROUTER_API_KEY')
    if k: return k
    try:
        from google.colab import userdata
        k = userdata.get('OPENROUTER_API_KEY')
        if k: return k
    except Exception:
        pass
    return getpass.getpass('Paste your OpenRouter API key (hidden): ')

API_KEY = _get_openrouter_key()
client  = OpenAI(base_url='https://openrouter.ai/api/v1', api_key=API_KEY)
GEN_MODEL = 'openai/gpt-4o-mini'      # TRY THIS (knob): any model your key can reach
TOKENS = {'in': 0, 'out': 0}

def generate(prompt, system=None, max_new_tokens=300, temperature=0.0):
    msgs = ([{'role': 'system', 'content': system}] if system else []) + \
           [{'role': 'user', 'content': prompt}]
    r = client.chat.completions.create(model=GEN_MODEL, messages=msgs,
                                       temperature=temperature, max_tokens=max_new_tokens)
    if getattr(r, 'usage', None):
        TOKENS['in']  += r.usage.prompt_tokens
        TOKENS['out'] += r.usage.completion_tokens
    return r.choices[0].message.content.strip()

print('connected ·', generate('Reply with exactly the word: ready', max_new_tokens=6))

Paste your OpenRouter API key (hidden): ··········
connected · ready


### Check your remaining credit  —  free, this call is not billed
Your allocation is **fixed and will not be topped up**, so watch it rather than discovering it at
the end. `usage` is what you have spent; `limit` is your cap; the difference is what is left.
Run this at the start of a working session and again at the end.

In [6]:
import urllib.request, json as _json

def credit_check():
    """Ask OpenRouter what is left on your key. Costs nothing."""
    try:
        req = urllib.request.Request('https://openrouter.ai/api/v1/key',
                                     headers={'Authorization': f'Bearer {API_KEY}'})
        d = _json.load(urllib.request.urlopen(req, timeout=20))['data']
        used, cap = d.get('usage'), d.get('limit')
        if cap is None:
            print(f'spent so far: ${used:.4f}   (no cap set on this key)')
        else:
            print(f'spent ${used:.4f} of ${cap:.2f}  ->  ${cap - used:.4f} remaining')
        rl = d.get('rate_limit') or {}
        if rl:
            print(f"rate limit: {rl.get('requests')} requests per {rl.get('interval')}")
    except Exception as e:
        print('could not check credit (not a problem, carry on):', str(e)[:120])

credit_check()

spent $0.0157 of $10.00  ->  $9.9843 remaining
rate limit: -1 requests per 10s


---
## 0 · Five words you need — *start here if you don't code*

| Word | In plain words |
|---|---|
| **Embedding** | A piece of text turned into a list of numbers, arranged so that texts with similar *meaning* end up near each other. |
| **Chunk** | A small slice of a document. We search chunks, not whole documents, so what comes back is the relevant *paragraph* rather than the whole manual. |
| **Retrieval** | Finding the chunks nearest the question. Pure arithmetic — no model involved, no understanding. |
| **Top-k** | How many chunks we fetch. Too few and the answer is missing; too many and the real answer drowns in noise. |
| **Grounding** | Making the model answer **only** from the retrieved text, and say so when the text does not cover it. |

**RAG in one line:** *embed your documents → retrieve the ones nearest the question → let a model
answer from them.* **The model is rented; the retrieval is yours.** That is why retrieval is where
almost every failure lives — and why the failure you document is the interesting part.

---
## 1 · Your documents  ### REPLACE ME

**10 to 30 documents.** Each a short paragraph — a few sentences is fine.

**The one rule that matters:** they must contain things a model *could not already know*. If your
documents are summaries of Wikipedia, the model answers correctly with retrieval switched off, and
you have proved nothing. Good sources: your own notes, internal procedures, club or team records,
a rulebook nobody has digitised, your family's version of anything, recent local information,
specifics with names and numbers in them.

**Test for this later, in Section 4.** We ask the model each question with the documents removed.
If it still gets them right, your document set is too generic — go back and make it specific.

In [7]:
### REPLACE ME — your documents. 10-30 short paragraphs from YOUR domain.
DOCS = [
    "For beginners, the correct grip pressure for badminton is similar to holding a bird gently. Squeezing too tight causes wrist stiffness and reduces shot control.",
    "The standard ready stance for badminton footwork: feet shoulder-width apart, knees slightly bent, weight on the balls of the feet, racket held at chest height.",
    "Split step timing: perform the small hop just before your opponent makes contact with the shuttle, not after they hit it. This improves reaction speed.",
    "Beginner backcourt forehand clear uses a full shoulder rotation, not just arm strength. Hit the shuttle at the highest point in front of your body.",
    "For new players, recommended racket weight is 85-90 grams. Heavier rackets generate more power but cause faster arm fatigue during long sessions.",
    "Beginner string tension should be 20-22 lbs. Higher tension gives better accuracy but reduces forgiveness and is more likely to cause wrist soreness.",
    "Badminton shoes have non-marking gum rubber soles designed for grip on indoor courts. Running shoes are unsuitable because they lack lateral support and can slip.",
    "To fix a net shot that hits the tape: reduce wrist power, contact the shuttle at a higher point, and brush the shuttle cork gently instead of pushing it.",
    "When moving to the forehand rear corner, use a cross-over step first, then adjust with side steps. Do not run backwards facing the net.",
    "After every shot, return to the center of the court (T-junction area) to maintain balanced coverage for your opponent's next return.",
    "overgrips should be replaced after 10-15 hours of play. Signs of wear include slipperiness, visible fraying and loss of original texture.",
    "For defensive lifts, contact the shuttle low and use a relaxed wrist flick. Hitting it too high and short gives your opponent an easy smash opportunity.",
    "Side shuffle steps are used for wide side-to-side movement. Cross steps are for fast long-distance movement to rear corners.",
    "Feather shuttlecocks break faster in cold dry air. Humid conditions make feathers softer and more durable but slower in flight.",
    "Backhand grip change: rotate the racket with your thumb and index finger, do not loosen your whole hand and re-grip from scratch.",
    "Lunging to the net: land on your heel first, then roll to the forefoot. Keep your knee aligned with your toes to avoid joint injury.",
    "Beginners should start with medium-flex rackets. Stiff rackets require fast swing speed to generate power and are harder to control.",
    "Wrist supports are not recommended for regular beginner training. They limit natural wrist movement and weaken your own wrist strength development.",
    "The main cause of shuttle going out on forehand clears is hitting the shuttle too late, behind your shoulder line. Move earlier to hit in front.",
    "To improve footwork speed, practice shadow footwork drills for 15 minutes per session. Focus on rhythm rather than maximum speed at first.",
]
print(f'{len(DOCS)} documents')
assert 10 <= len(DOCS) <= 40, 'The brief asks for 10-30 documents.'

20 documents


### Chunking
Long documents get cut into overlapping slices before embedding. Why: one embedding holds one
*average* meaning, so a long multi-topic document becomes a blurry vector that matches everything
a little and nothing well. Small chunks keep each vector about a single idea. The **overlap** stops
an idea that straddles a boundary from being cut in half.

This is a knob you tune against your evidence, not a setting with a correct value.

In [8]:
# TRY THIS (knob): watch what changes in Section 3 when you move these.
CHUNK_WORDS = 45     # try 15 (tiny) or 300 (whole document = one chunk)
OVERLAP     = 12     # words shared with the next chunk

def chunk(text, size=CHUNK_WORDS, overlap=OVERLAP):
    words, out, i = text.split(), [], 0
    while i < len(words):
        out.append(' '.join(words[i:i+size]))
        if i + size >= len(words):
            break
        i += max(1, size - overlap)
    return out

CHUNKS = [(c, d_i) for d_i, d in enumerate(DOCS) for c in chunk(d)]
print(f'{len(DOCS)} docs -> {len(CHUNKS)} chunks (CHUNK_WORDS={CHUNK_WORDS}, OVERLAP={OVERLAP})')

20 docs -> 20 chunks (CHUNK_WORDS=45, OVERLAP=12)


### Embed and retrieve
The embedder runs locally — a small model downloaded once, no key, no cost. If the download fails
(no internet on the machine, or a locked-down environment), the fallback cell below uses **TF-IDF**
word-overlap instead. TF-IDF works, and it is a useful contrast: it matches *words*, not *meaning*,
so it fails on paraphrase. If you use the fallback, say so in your writeup — it changes which
failures you find.

In [9]:
EMBEDDER = 'sentence-transformers/all-MiniLM-L6-v2'
USING = None
try:
    from sentence_transformers import SentenceTransformer
    _m = SentenceTransformer(EMBEDDER)
    def embed(texts):
        v = _m.encode(list(texts), normalize_embeddings=True)
        return np.asarray(v)
    USING = f'local embeddings ({EMBEDDER}) — matches MEANING'
except Exception as e:
    print('embedder unavailable ->', str(e)[:120])
    from sklearn.feature_extraction.text import TfidfVectorizer
    _v = TfidfVectorizer().fit([c for c, _ in CHUNKS])
    def embed(texts):
        m = _v.transform(list(texts)).toarray()
        n = np.linalg.norm(m, axis=1, keepdims=True)
        return m / np.where(n == 0, 1, n)
    USING = 'TF-IDF fallback — matches WORDS, not meaning'

print('using:', USING)
MATRIX = embed([c for c, _ in CHUNKS])

TOP_K = 3            # TRY THIS (knob): 1 (brittle) or 8 (noisy)

def retrieve(question, k=TOP_K):
    q = embed([question])[0]
    scores = MATRIX @ q
    order = np.argsort(-scores)[:k]
    return [(CHUNKS[i][0], CHUNKS[i][1], float(scores[i])) for i in order]

for txt, d_i, s in retrieve('how often is the fiddle leaf fig watered in winter?'):
    print(f'  {s:.3f}  [doc {d_i}]  {txt[:80]}...')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

using: local embeddings (sentence-transformers/all-MiniLM-L6-v2) — matches MEANING
  0.166  [doc 13]  Feather shuttlecocks break faster in cold dry air. Humid conditions make feather...
  0.164  [doc 10]  overgrips should be replaced after 10-15 hours of play. Signs of wear include sl...
  0.073  [doc 5]  Beginner string tension should be 20-22 lbs. Higher tension gives better accurac...


### Answer from the retrieved text — and only from it
The instruction to say *"the notes do not say"* is the whole point. Without it the model fills gaps
from its training data and you cannot tell which sentences came from your documents. That is
**silent failure**: a confident, fluent, unsourced answer.

In [10]:
GROUNDED = (
    "Answer using ONLY the notes provided. "
    "If the notes do not contain the answer, reply exactly: The notes do not say. "
    "Do not use any other knowledge. Quote the phrase you relied on."
)

def answer(question, k=TOP_K, show=True):
    hits = retrieve(question, k)
    context = '\n\n'.join(f'[{i+1}] {t}' for i, (t, _, _) in enumerate(hits))
    out = generate(f'NOTES:\n{context}\n\nQUESTION: {question}', system=GROUNDED)
    if show:
        print(f'Q: {question}\nA: {out}\n   (top chunk score {hits[0][2]:.3f}, doc {hits[0][1]})\n')
    return out, hits

answer('When was the fiddle-leaf fig repotted, and into what?')

Q: When was the fiddle-leaf fig repotted, and into what?
A: The notes do not say.
   (top chunk score 0.159, doc 10)



('The notes do not say.',
 [('overgrips should be replaced after 10-15 hours of play. Signs of wear include slipperiness, visible fraying and loss of original texture.',
   10,
   0.15914419293403625),
  ('The standard ready stance for badminton footwork: feet shoulder-width apart, knees slightly bent, weight on the balls of the feet, racket held at chest height.',
   1,
   0.13950325548648834),
  ('Lunging to the net: land on your heel first, then roll to the forefoot. Keep your knee aligned with your toes to avoid joint injury.',
   15,
   0.1249145120382309)])

---
## 2 · Five questions it could not answer without your documents  ### REPLACE ME

Each question should have a **specific** answer that lives in your documents — a date, a name, a
number, a decision and its outcome. "What is a monstera?" is a bad question: the model knows that
already, so a correct answer proves nothing about your retrieval.

In [11]:
### REPLACE ME — five questions answerable ONLY from your documents.
QUESTIONS = [
   "How often should badminton players replace their overgrips?",
    "What string tension range is recommended for beginner badminton players?",
    "When should a player perform the split step during a badminton rally?",
    "What is the main reason a beginner's forehand clear often goes out of bounds?",
    "What racket weight range is suitable for new badminton players?"
]
for q in QUESTIONS:
    answer(q)

Q: How often should badminton players replace their overgrips?
A: "overgrips should be replaced after 10-15 hours of play."
   (top chunk score 0.611, doc 10)

Q: What string tension range is recommended for beginner badminton players?
A: The recommended string tension range for beginner badminton players is "20-22 lbs."
   (top chunk score 0.718, doc 5)

Q: When should a player perform the split step during a badminton rally?
A: "A player should perform the split step just before your opponent makes contact with the shuttle."
   (top chunk score 0.588, doc 2)

Q: What is the main reason a beginner's forehand clear often goes out of bounds?
A: The main reason a beginner's forehand clear often goes out of bounds is "hitting the shuttle too late, behind your shoulder line."
   (top chunk score 0.608, doc 18)

Q: What racket weight range is suitable for new badminton players?
A: The suitable racket weight range for new badminton players is 85-90 grams. "For new players, recommended racket

---
## 3 · Break it  ### REPLACE ME

**This section carries the marks.** You are looking for a question where retrieval brings back the
wrong passage — not where the model writes a clumsy sentence.

Four kinds of break that reliably work. Try all four; keep the one that fails most interestingly.

| Kind | What to try |
|---|---|
| **Vocabulary mismatch** | Ask using a word that never appears in your documents but means the same thing. Devastating for TF-IDF; embeddings usually survive it. |
| **Two documents needed** | Ask something that requires combining facts from two documents. Top-k retrieval fetches the *most similar* chunks, which is not the same as *the set that answers the question*. |
| **Distractor** | Ask about a topic covered in several documents where only one is relevant. Watch the right chunk get outvoted. |
| **Absent but plausible** | Ask something your documents nearly, but do not quite, cover. Does it say "the notes do not say", or does it invent? |

**Diagnose, do not just report.** "It got it wrong" is a Developing answer. "The right chunk ranked
fourth with score 0.31 while an irrelevant chunk scored 0.44 because both contain the word
*water*" is a Proficient one.

In [12]:
### REPLACE ME — your breaking attempts.
BREAKERS = [
   ("vocabulary mismatch", "How often should badminton grip tape be replaced?"),
    ("two documents needed", "What is the recommended racket weight and string tension for new players?"),
    ("distractor", "What causes wrist soreness for beginner badminton players?"),
    ("absent but plausible", "How long does a badminton racket usually last?")
]
for kind, q in BREAKERS:
    print(f'--- {kind.upper()} ---')
    out, hits = answer(q)
    print('   retrieved:')
    for t, d_i, s in hits:
        print(f'     {s:.3f} [doc {d_i}] {t[:70]}...')
    print()

--- VOCABULARY MISMATCH ---
Q: How often should badminton grip tape be replaced?
A: "overgrips should be replaced after 10-15 hours of play."
   (top chunk score 0.500, doc 10)

   retrieved:
     0.500 [doc 10] overgrips should be replaced after 10-15 hours of play. Signs of wear ...
     0.467 [doc 7] To fix a net shot that hits the tape: reduce wrist power, contact the ...
     0.444 [doc 0] For beginners, the correct grip pressure for badminton is similar to h...

--- TWO DOCUMENTS NEEDED ---
Q: What is the recommended racket weight and string tension for new players?
A: The recommended racket weight for new players is "85-90 grams" and the beginner string tension should be "20-22 lbs."
   (top chunk score 0.744, doc 5)

   retrieved:
     0.744 [doc 5] Beginner string tension should be 20-22 lbs. Higher tension gives bett...
     0.742 [doc 4] For new players, recommended racket weight is 85-90 grams. Heavier rac...
     0.663 [doc 16] Beginners should start with medium-flex racke

### Diagnose the one you will submit
Fill this in **in prose**, from the scores printed above. This is your documented failure.

In [13]:
FAILURE_DIAGNOSIS = """
### REPLACE ME

Question that failed: What causes wrist soreness for beginner badminton players?
What the system answered: The notes do not say.
What the correct answer was: High string tension can lead to wrist soreness for new‑level players.
Which chunk SHOULD have been retrieved, and what did it score: The correct information was stored inside doc 5, which was not returned within the top‑3 retrieved results.
What was retrieved instead, and what did it score: Doc 0 with score 0.686 about grip pressure, doc 17 with score 0.525 about wrist supports, and doc 1 with score 0.471 about ready stance footwork.
WHY (the mechanism — vocabulary, chunk size, top‑k, a distractor, or genuinely absent): This failure was caused by a distractor problem. The retrieved chunks share related keywords such as wrist and grip, so they achieved high semantic similarity scores, even though they do not contain the factual answer. The correct chunk was out‑ranked and pushed outside the top‑k retrieval window.
What you would change to fix it, and what that change would cost you elsewhere: Increasing the top‑k retrieval number could bring more candidate chunks and improve the chance of capturing the correct document. The trade‑off would be higher latency and slightly higher API cost, as more text would be sent to the large‑language‑model prompt.
"""

print(FAILURE_DIAGNOSIS)


### REPLACE ME

Question that failed: What causes wrist soreness for beginner badminton players?
What the system answered: The notes do not say.
What the correct answer was: High string tension can lead to wrist soreness for new‑level players.
Which chunk SHOULD have been retrieved, and what did it score: The correct information was stored inside doc 5, which was not returned within the top‑3 retrieved results.
What was retrieved instead, and what did it score: Doc 0 with score 0.686 about grip pressure, doc 17 with score 0.525 about wrist supports, and doc 1 with score 0.471 about ready stance footwork.
WHY (the mechanism — vocabulary, chunk size, top‑k, a distractor, or genuinely absent): This failure was caused by a distractor problem. The retrieved chunks share related keywords such as wrist and grip, so they achieved high semantic similarity scores, even though they do not contain the factual answer. The correct chunk was out‑ranked and pushed outside the top‑k retrieval window.


---
## 4 · Two checks worth running before you write

**Check 1 — is retrieval doing anything?** Ask the same five questions with the documents removed.
If the model still answers correctly, your documents are too generic and Part 2 has not been
demonstrated. Fix the documents.

**Check 2 — does the grounding instruction hold?** A system that invents when it should decline is
worse than one that declines too often, because you cannot see it happening.

In [14]:
print(answer("How often should badminton players replace their overgrips?"))
print(answer("What string tension range is recommended for beginner badminton players?"))
print('=== CHECK 1: same questions, NO documents ===')
for q in QUESTIONS[:5]:
    print(f'Q: {q}\nA: {generate(q, system="Answer in one short sentence.", max_new_tokens=80)}\n')

print('=== CHECK 2: something your documents definitely do not cover ===')
answer('What is the postal code of the flat?')

Q: How often should badminton players replace their overgrips?
A: "overgrips should be replaced after 10-15 hours of play."
   (top chunk score 0.611, doc 10)

('"overgrips should be replaced after 10-15 hours of play."', [('overgrips should be replaced after 10-15 hours of play. Signs of wear include slipperiness, visible fraying and loss of original texture.', 10, 0.6114802360534668), ('The standard ready stance for badminton footwork: feet shoulder-width apart, knees slightly bent, weight on the balls of the feet, racket held at chest height.', 1, 0.4762474596500397), ('For beginners, the correct grip pressure for badminton is similar to holding a bird gently. Squeezing too tight causes wrist stiffness and reduces shot control.', 0, 0.4660564064979553)])
Q: What string tension range is recommended for beginner badminton players?
A: The recommended string tension range for beginner badminton players is "20-22 lbs."
   (top chunk score 0.718, doc 5)

('The recommended string tension r

('The notes do not say.',
 [('The standard ready stance for badminton footwork: feet shoulder-width apart, knees slightly bent, weight on the balls of the feet, racket held at chest height.',
   1,
   0.05063080042600632),
  ('Lunging to the net: land on your heel first, then roll to the forefoot. Keep your knee aligned with your toes to avoid joint injury.',
   15,
   0.041755691170692444),
  ("After every shot, return to the center of the court (T-junction area) to maintain balanced coverage for your opponent's next return.",
   9,
   0.0375271774828434)])

---
## 5 · YOUR TURN

1. Replace `DOCS` with 10–30 documents from your domain that contain things a model cannot know.
2. Replace `QUESTIONS` with five that are answerable only from them. Run and read every answer.
3. Run Check 1. If the model answers without your documents, go back to step 1.
4. Work through all four `BREAKERS`. Keep the most interesting failure and diagnose it properly.
5. Try `CHUNK_WORDS`, `OVERLAP` and `TOP_K` at extremes. Note which failures appear and disappear.

### What your ~400 words must contain

- What your documents are, and why a model could not answer these questions without them.
- The five worked examples (the notebook output is enough; the words are for what they show).
- **The failure, diagnosed** — which chunk should have won, which did, the scores, and the
  mechanism.
- What you would change to fix it, and **what that fix would break** — bigger chunks improve
  context and dilute matching; a larger top-k improves recall and adds noise and tokens. There is
  no free setting, and saying so is the postgraduate move.

In [ ]:
print(f'tokens this session: {TOKENS["in"]} in / {TOKENS["out"]} out')